In [2]:
from pathlib import Path
import sys

start = Path.cwd().resolve()
PROJECT_ROOT = next((folder for folder in (start, *start.parents)
                     if (folder / "replication_paths.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open Jupyter inside the replication project directory.")
for source_dir in (PROJECT_ROOT, PROJECT_ROOT / "01_taxonomy", PROJECT_ROOT / "03_judge_prompts",
                   PROJECT_ROOT / "04_llm_evaluation"):
    if str(source_dir) not in sys.path:
        sys.path.insert(0, str(source_dir))

from replication_paths import RESULTS_DIR, load_sample
from llm_runner import run_evaluations, experiment_status
from IPython.display import display
import pandas as pd

model_name = "gemini"
model = "google/gemini-3.8-flash"
api_base_url = "https://openrouter.ai/api/v1"
api_key_variable = "OPENROUTER_API_KEY"
use_json_mode = False
max_output_tokens = 16384

# Set True only when you explicitly intend to make new, paid API requests.
RUN_API_CALLS = False

df_sample = load_sample()
assert df_sample["prompt_id"].is_unique
print(f"Loaded {len(df_sample)} prompts; archived results:")
display(pd.DataFrame(experiment_status(RESULTS_DIR, model_name)))

Loaded 260 prompts; archived results:


,technique,saved_classifications,logged_errors
0,zero_shot,260,0
1,few_shot,259,0
2,mv3,259,0


In [3]:
client = None
if RUN_API_CALLS:
    import os
    from getpass import getpass
    from openai import OpenAI
    api_key = os.getenv(api_key_variable) or getpass(f"{api_key_variable}: ").strip()
    if not api_key:
        raise ValueError("An API key is required for paid evaluation calls")
    client = OpenAI(api_key=api_key, base_url=api_base_url)

In [6]:
if RUN_API_CALLS:
    run_evaluations(
        df_sample, client=client, api_model=model, model_name=model_name,
        results_dir=RESULTS_DIR, technique="zero_shot",
        max_tokens=max_output_tokens, json_mode=use_json_mode,
    )
else:
    print("Zero-shot: existing records preserved (API calls disabled)")

Zero-shot: existing records preserved (API calls disabled)


In [7]:
if RUN_API_CALLS:
    run_evaluations(
        df_sample, client=client, api_model=model, model_name=model_name,
        results_dir=RESULTS_DIR, technique="few_shot",
        max_tokens=max_output_tokens, json_mode=use_json_mode,
    )
else:
    print("Few-shot: existing records preserved (API calls disabled)")

Few-shot: existing records preserved (API calls disabled)


In [8]:
if RUN_API_CALLS:
    run_evaluations(
        df_sample, client=client, api_model=model, model_name=model_name,
        results_dir=RESULTS_DIR, technique="mv3",
        max_tokens=max_output_tokens, json_mode=use_json_mode,
    )
else:
    print("Best-of-3 (majority vote): existing records preserved (API calls disabled)")

Best-of-3 (majority vote): existing records preserved (API calls disabled)


In [9]:
display(pd.DataFrame(experiment_status(RESULTS_DIR, model_name)))

,technique,saved_classifications,logged_errors
0,zero_shot,260,0
1,few_shot,259,0
2,mv3,259,0
